In [0]:
BASE_DIR = '/Volumes/sales/landing/sales/OrderItems/'
CHECKPOINT_DIR = '/Volumes/sales/landing/sales/CheckPoint_Directory/'
CHECKPOINT_LOCATION = f'{CHECKPOINT_DIR}/OrderItems'
CATALOG = 'Sales'
SCHEMA = 'Bronze'
TARGET_TABLE = 'OrderItems'
print(f'checkpoint_location : {CHECKPOINT_LOCATION}')
print(f'BASE_LOCATION = {BASE_DIR}')


In [0]:
OI_schema = '''order_id STRING , 
               order_item_id INT, 
               product_id STRING, 
               seller_id STRING, 
               shipping_limit_date TIMESTAMP_NTZ, 
               price DOUBLE, 
               freight_value DOUBLE'''

In [0]:
OI_df = spark.read.format('csv')\
                  .option('header', 'true')\
                  .schema(OI_schema)\
                  .load(f'{BASE_DIR}')
display(OI_df.limit(3))
OI_df.printSchema()

#### LOAD THE DATA INTO BRONZE LAYER
> Order_Sales . Bronze . printSchema

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
OI_df = (
            spark.readStream.format('cloudFiles')
                 .option('cloudFiles.format', 'csv')
                 .option('cloudFiles.schemaEvolutionMode', 'rescue')
                 .option('header', 'true')
                 .schema(OI_schema)
                 .load(f'{BASE_DIR}')
        )
OI_trans_df = OI_df.select(
    "order_id", 
    "order_item_id", 
    "product_id", 
    "seller_id", 
    "shipping_limit_date",
    "price" ,
    "freight_value",
    "_metadata.file_name",
    "_metadata.file_path",).withColumn('load_timestamp', current_timestamp())

#OI_trans_df.printSchema()
#OI_trans_df.limit(3).display(checkpointLocation=f'{CHECKPOINT_DIR}/OrderItems_display')

In [0]:
qry = (
    OI_trans_df.writeStream.format('delta')
    .outputMode('append')
    .option('checkpointLocation', CHECKPOINT_LOCATION)
    .trigger(availableNow=True)
    .toTable(f'{CATALOG}.{SCHEMA}.{TARGET_TABLE}')
)
qry.awaitTermination()